[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/056.%20The%20Safety%20Stock%20%26%20Reorder%20Point%20%28Q%2Cr%29%20Policy%20Problem/P56-Tier-3_executed.ipynb)

---


# 56. The Safety Stock and Reorder Point \((Q,r)\) Policy Problem

## Tier 3: Particle Swarm Optimization

### Purpose and inputs

This notebook uses particle swarm optimization (PSO) to search for an order quantity \(Q\) and reorder point \(r\) that minimize a modeled annual cost. The example represents CardioStabil, an item with uncertain daily demand and uncertain replenishment lead time. Inputs include annual demand, demand and lead-time variability, unit and ordering costs, a stockout cost, and a target cycle service level. The outputs are a candidate policy, its modeled cost and service level, and summaries of the swarm's search.

PSO is a stochastic population-based search method. Each particle is a candidate point \((Q,r)\) with a velocity that moves it through the search space. Particles learn from their own best-known positions and the swarm's best-known position. This method can search a nonlinear objective without requiring derivatives, but it does **not** guarantee that the global optimum will be found. Because the search uses random initialization and random coefficients, results can differ between runs.

### Modeling assumptions and scope

- Daily demand is modeled as independent observations with a common mean and standard deviation. Lead time is independent of daily demand and has a specified mean and standard deviation.
- Demand during lead time is approximated by a normal distribution using its mean and variance. The resulting service level is a **cycle service level**: the probability that demand during lead time does not exceed the reorder point.
- The annual stockout-cost term is a simple proxy: the probability of a stockout cycle is multiplied by annual demand divided by four, then by the cost per modeled stockout. It is not an exact expected-shortage calculation.
- A soft service-level penalty is included in the objective. The target is therefore not a hard feasibility constraint; a candidate below the target can still be evaluated, but it incurs a penalty.
- The search is restricted to the stated bounds for \(Q\) and \(r\). The model does not represent such details as order quantities restricted to integers, backorder timing, quantity discounts, perishability, or capacity constraints.

### Cost model

For a candidate policy, the notebook estimates annual holding, ordering, stockout, and service-penalty costs:

$$
\begin{aligned}
\mu_L &= \mu_d L, \\
\sigma_L &= \sqrt{L\sigma_d^2 + \mu_d^2\sigma_{LT}^2}, \\
SS &= \max(0,r-\mu_L), \\
SL(r) &= \Phi\!\left(\frac{r-\mu_L}{\sigma_L}\right), \\
C_H(Q,r) &= \left(\frac{Q}{2}+SS\right)c h, \\
C_O(Q) &= \frac{D}{Q}K, \\
C_S(r) &= \left(1-SL(r)\right)\frac{D}{4}c_s, \\
C_P(r) &= 50{,}000\max\left(0,SL_{\text{target}}-SL(r)\right), \\
C(Q,r) &= C_H(Q,r)+C_O(Q)+C_S(r)+C_P(r).
\end{aligned}
$$

Here, \(D\) is annual demand; \(\mu_d\) and \(\sigma_d\) are mean daily demand and its standard deviation; \(L\) and \(\sigma_{LT}\) are mean lead time and its standard deviation; \(\mu_L\) and \(\sigma_L\) are the approximated mean and standard deviation of demand during lead time; \(SS\) is safety stock; \(c\) is unit cost; \(h\) is the annual holding-rate fraction; \(K\) is the cost per order; and \(c_s\) is the modeled cost per stockout. \(\Phi\) is the standard normal cumulative distribution function. The code implements these quantities in `calculate_cost_components` and `_calculate_service_level`; the component sum is the fitness minimized by PSO.

The variance expression follows from the stated independent-demand and random-lead-time assumptions. If lead time is zero, or the resulting demand variance is zero, the service probability is evaluated deterministically rather than by dividing by zero.

### PSO search and interpretation

Let particle \(i\) have position \(x_i=(Q_i,r_i)\) and velocity \(v_i\). At iteration \(t\), the update is

$$
\begin{aligned}
v_i^{t+1} &= w_t v_i^t + c_1 u_1(p_i-x_i^t)+c_2 u_2(g-x_i^t), \\
x_i^{t+1} &= x_i^t+v_i^{t+1}, \\
w_t &= w_{\max}-(w_{\max}-w_{\min})\frac{t}{T}.
\end{aligned}
$$

The personal best \(p_i\) is the best position yet visited by particle \(i\), and the global best \(g\) is the best position found by any particle. The independent random coefficients \(u_1,u_2\) are sampled uniformly from \([0,1]\); \(c_1,c_2\) control the pull toward personal and swarm experience; and the inertia weight \(w_t\) decreases linearly to balance movement with refinement. After each move, the code clips both coordinates to their search bounds and evaluates the cost. In the implementation, `position`, `velocity`, `best_position`, and `global_best_position` represent these quantities.

The algorithm first evaluates the initial swarm, then repeats particle updates for at most `max_iterations`. It stops early if the relative improvement in the best cost over the recent history is below the stated threshold. With swarm size \(S\), at most \(T\) iterations, and a constant-cost evaluation, the search requires \(O(ST)\) fitness evaluations; this describes the computational work, not a guarantee of solution quality. The reported candidate is the best position found by this stochastic search, not a proof of global optimality.

In [ ]:
# Import the numerical, statistical, data, and plotting tools used by the notebook.
import random
from dataclasses import dataclass
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for particle swarm optimization.")

In [ ]:
@dataclass
class DemandParameters:
    """Demand and lead-time inputs for the inventory model."""

    annual_demand: float
    daily_std: float
    lead_time: float
    lead_time_std: float


@dataclass
class CostParameters:
    """Cost inputs for the inventory model."""

    unit_cost: float
    holding_rate: float
    ordering_cost: float
    stockout_cost: float


class PSOInventoryOptimizer:
    """Use particle swarm optimization to search a bounded (Q, r) space."""

    def __init__(
        self,
        demand_params: DemandParameters,
        cost_params: CostParameters,
        service_level: float,
        swarm_size: int = 30,
        max_iterations: int = 100,
    ):
        """Store model inputs and initialize the PSO search."""
        if demand_params.annual_demand <= 0:
            raise ValueError("Annual demand must be positive.")
        if demand_params.daily_std < 0 or demand_params.lead_time_std < 0:
            raise ValueError("Demand and lead-time standard deviations cannot be negative.")
        if demand_params.lead_time < 0:
            raise ValueError("Lead time cannot be negative.")
        if min(
            cost_params.unit_cost,
            cost_params.holding_rate,
            cost_params.ordering_cost,
            cost_params.stockout_cost,
        ) < 0:
            raise ValueError("Cost parameters cannot be negative.")
        if not 0 < service_level < 1:
            raise ValueError("The target service level must be strictly between 0 and 1.")
        if swarm_size < 1 or max_iterations < 1:
            raise ValueError("Swarm size and maximum iterations must be positive.")

        self.demand_params = demand_params
        self.cost_params = cost_params
        self.service_level = service_level
        self.swarm_size = swarm_size
        self.max_iterations = max_iterations

        self.daily_demand = demand_params.annual_demand / 365
        self.z_score = stats.norm.ppf(service_level)

        # A decreasing inertia weight shifts the search from broad movement toward refinement.
        self.w_max, self.w_min = 0.9, 0.4
        self.c1, self.c2 = 2.0, 2.0

        self.q_bounds = [10, demand_params.annual_demand / 2]
        mean_lead_demand = self.daily_demand * demand_params.lead_time
        self.r_bounds = [mean_lead_demand, mean_lead_demand * 3]

        if self.q_bounds[1] < self.q_bounds[0]:
            raise ValueError("The annual demand does not support the specified Q search bounds.")

        self.initialize_swarm()

    def initialize_swarm(self):
        """Create particles with positions and velocities inside the search space."""
        self.particles = []

        for _ in range(self.swarm_size):
            q = random.uniform(self.q_bounds[0], self.q_bounds[1])
            r = random.uniform(self.r_bounds[0], self.r_bounds[1])
            position = np.array([q, r], dtype=float)

            velocity = np.array(
                [
                    random.uniform(-0.1 * self.q_bounds[1], 0.1 * self.q_bounds[1]),
                    random.uniform(-0.1 * self.r_bounds[1], 0.1 * self.r_bounds[1]),
                ],
                dtype=float,
            )

            self.particles.append(
                {
                    "position": position,
                    "velocity": velocity,
                    "best_position": position.copy(),
                    "best_fitness": float("inf"),
                    "fitness": float("inf"),
                }
            )

        # The center is only a placeholder until the initial particles are evaluated.
        self.global_best_position = np.array(
            [
                (self.q_bounds[0] + self.q_bounds[1]) / 2,
                (self.r_bounds[0] + self.r_bounds[1]) / 2,
            ],
            dtype=float,
        )
        self.global_best_fitness = float("inf")

        print(f"Swarm initialized with {self.swarm_size} particles.")
        print(f"Q bounds: [{self.q_bounds[0]:.1f}, {self.q_bounds[1]:.1f}]")
        print(f"r bounds: [{self.r_bounds[0]:.1f}, {self.r_bounds[1]:.1f}]")


print("Demand and cost parameters, and the PSO optimizer, are defined.")

In [ ]:
class PSOInventoryOptimizer(PSOInventoryOptimizer):
    """Add the inventory objective and service-level calculation."""

    def _calculate_service_level(self, r: float) -> float:
        """Estimate cycle service level from demand during lead time."""
        expected_demand = self.daily_demand * self.demand_params.lead_time
        demand_variance = (
            self.demand_params.lead_time * self.demand_params.daily_std**2
        )
        lead_time_variance = (
            self.daily_demand**2 * self.demand_params.lead_time_std**2
        )
        combined_std = np.sqrt(demand_variance + lead_time_variance)

        # With no demand uncertainty, service is determined by whether r covers certain lead-time demand.
        if np.isclose(combined_std, 0.0):
            return float(r >= expected_demand)

        z_value = (r - expected_demand) / combined_std
        return float(stats.norm.cdf(z_value))

    def calculate_cost_components(self, q: float, r: float) -> Dict[str, float]:
        """Return the modeled annual cost components for a candidate policy."""
        if q <= 0:
            raise ValueError("Order quantity Q must be positive.")

        expected_lead_demand = self.daily_demand * self.demand_params.lead_time
        safety_stock = max(0.0, r - expected_lead_demand)
        cycle_stock = q / 2

        holding_cost = (
            (cycle_stock + safety_stock)
            * self.cost_params.unit_cost
            * self.cost_params.holding_rate
        )
        ordering_cost = (
            self.demand_params.annual_demand / q
        ) * self.cost_params.ordering_cost

        service_level_achieved = self._calculate_service_level(r)
        stockout_probability = max(0.0, 1.0 - service_level_achieved)
        expected_stockouts = (
            stockout_probability * self.demand_params.annual_demand / 4
        )
        stockout_cost = expected_stockouts * self.cost_params.stockout_cost
        service_penalty = max(
            0.0, self.service_level - service_level_achieved
        ) * 50000

        return {
            "holding_cost": holding_cost,
            "ordering_cost": ordering_cost,
            "stockout_cost": stockout_cost,
            "service_penalty": service_penalty,
            "safety_stock": safety_stock,
            "service_level_achieved": service_level_achieved,
        }

    def calculate_fitness(self, q: float, r: float) -> float:
        """Return total modeled annual cost; PSO seeks to minimize this value."""
        components = self.calculate_cost_components(q, r)
        return sum(
            components[name]
            for name in (
                "holding_cost",
                "ordering_cost",
                "stockout_cost",
                "service_penalty",
            )
        )


print("The modeled cost function and cycle service-level calculation are ready.")

In [ ]:
class PSOInventoryOptimizer(PSOInventoryOptimizer):
    """Add particle movement and the iterative PSO search."""

    def update_particle(self, particle: Dict, iteration: int):
        """Move one particle, enforce bounds, and update its best-known positions."""
        inertia = self.w_max - (
            (self.w_max - self.w_min) * iteration / self.max_iterations
        )
        random_cognitive = random.random()
        random_social = random.random()

        cognitive = self.c1 * random_cognitive * (
            particle["best_position"] - particle["position"]
        )
        social = self.c2 * random_social * (
            self.global_best_position - particle["position"]
        )
        particle["velocity"] = (
            inertia * particle["velocity"] + cognitive + social
        )
        particle["position"] += particle["velocity"]

        # Clipping keeps each candidate feasible in the two-dimensional search space.
        particle["position"][0] = np.clip(
            particle["position"][0], *self.q_bounds
        )
        particle["position"][1] = np.clip(
            particle["position"][1], *self.r_bounds
        )

        particle["fitness"] = self.calculate_fitness(
            particle["position"][0], particle["position"][1]
        )

        if particle["fitness"] < particle["best_fitness"]:
            particle["best_fitness"] = particle["fitness"]
            particle["best_position"] = particle["position"].copy()

        if particle["fitness"] < self.global_best_fitness:
            self.global_best_fitness = particle["fitness"]
            self.global_best_position = particle["position"].copy()

    def optimize(self) -> Dict:
        """Evaluate the initial swarm and run the bounded PSO iteration loop."""
        fitness_history = []

        print("Evaluating the initial swarm...")
        for particle in self.particles:
            particle["fitness"] = self.calculate_fitness(
                particle["position"][0], particle["position"][1]
            )

            if particle["fitness"] < particle["best_fitness"]:
                particle["best_fitness"] = particle["fitness"]
                particle["best_position"] = particle["position"].copy()

            if particle["fitness"] < self.global_best_fitness:
                self.global_best_fitness = particle["fitness"]
                self.global_best_position = particle["position"].copy()

        fitness_history.append(self.global_best_fitness)
        print("Starting PSO iterations...")

        converged = False
        iterations_run = 0

        for iteration in range(self.max_iterations):
            for particle in self.particles:
                self.update_particle(particle, iteration)

            iterations_run = iteration + 1
            fitness_history.append(self.global_best_fitness)

            # Stop when the relative improvement over the recent history is small.
            if iteration > 10:
                reference_cost = fitness_history[-10]
                recent_improvement = (
                    reference_cost - fitness_history[-1]
                ) / max(abs(reference_cost), np.finfo(float).tiny)
                if recent_improvement < 0.001:
                    converged = True
                    print(f"Early stopping at iteration {iterations_run}.")
                    break

            if iterations_run % 20 == 0:
                print(
                    f"Iteration {iterations_run}: "
                    f"best modeled cost = ${self.global_best_fitness:,.2f}"
                )

        optimal_q, optimal_r = self.global_best_position
        components = self.calculate_cost_components(optimal_q, optimal_r)
        total_cost = self.calculate_fitness(optimal_q, optimal_r)

        return {
            "Q": optimal_q,
            "r": optimal_r,
            "safety_stock": components["safety_stock"],
            "total_cost": total_cost,
            "service_level_achieved": components["service_level_achieved"],
            "fitness_history": fitness_history,
            "iterations": iterations_run,
            "converged": converged,
        }


print("Particle updates and the PSO optimization loop are ready.")

In [ ]:
# Define the CardioStabil demand, lead-time, and cost inputs.
demand_params = DemandParameters(
    annual_demand=12000,
    daily_std=10,
    lead_time=14,
    lead_time_std=3,
)

cost_params = CostParameters(
    unit_cost=450,
    holding_rate=0.25,
    ordering_cost=200,
    stockout_cost=2000,
)

service_level = 0.995

print("=== INITIALIZING THE PSO OPTIMIZER ===")
optimizer = PSOInventoryOptimizer(
    demand_params=demand_params,
    cost_params=cost_params,
    service_level=service_level,
    swarm_size=40,
    max_iterations=150,
)

print(f"\nTarget cycle service level: {service_level * 100:.1f}%")
print(f"Mean daily demand: {optimizer.daily_demand:.1f} units")
print(f"Target normal quantile: {optimizer.z_score:.3f}")

print("\n=== RUNNING THE PSO SEARCH ===")
result = optimizer.optimize()

print("\n=== BEST CANDIDATE FOUND ===")
print(f"Early stopping criterion met: {result['converged']}")
print(f"Iterations performed: {result['iterations']}")
print(f"Order quantity Q: {result['Q']:.0f} units")
print(f"Reorder point r: {result['r']:.0f} units")
print(f"Safety stock: {result['safety_stock']:.0f} units")
print(f"Modeled annual cost: ${result['total_cost']:,.2f}")
print(
    f"Cycle service level achieved: "
    f"{result['service_level_achieved']:.4f} "
    f"({result['service_level_achieved'] * 100:.2f}%)"
)
print(
    f"Target cycle service level: {service_level:.4f} "
    f"({service_level * 100:.2f}%)"
)
print(
    f"Service-level gap (achieved minus target): "
    f"{(result['service_level_achieved'] - service_level) * 100:+.3f} percentage points"
)

# These checks link the reported candidate to its bounds and the stated objective.
assert optimizer.q_bounds[0] <= result["Q"] <= optimizer.q_bounds[1]
assert optimizer.r_bounds[0] <= result["r"] <= optimizer.r_bounds[1]
assert np.isclose(
    result["total_cost"],
    optimizer.calculate_fitness(result["Q"], result["r"]),
)


In [ ]:
def create_pso_visualization(result: Dict, optimizer: PSOInventoryOptimizer):
    """Plot search history, final swarm positions, service, and cost components."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Particle Swarm Search: CardioStabil Inventory Policy",
        fontsize=16,
        fontweight="bold",
    )

    # The history includes the initial swarm evaluation at iteration zero.
    fitness_history = result["fitness_history"]
    iterations = range(len(fitness_history))
    ax1.plot(iterations, fitness_history, color="blue", linewidth=2, label="Best cost")
    ax1.fill_between(iterations, fitness_history, alpha=0.2, color="blue")
    ax1.set_xlabel("Iteration (initial evaluation is 0)")
    ax1.set_ylabel("Best modeled annual cost ($)")
    ax1.set_title("Best-Cost History")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    if result["converged"]:
        ax1.axvline(
            x=result["iterations"],
            color="red",
            linestyle="--",
            label="Early stopping",
        )
        ax1.legend()

    q_positions = [particle["position"][0] for particle in optimizer.particles]
    r_positions = [particle["position"][1] for particle in optimizer.particles]
    fitnesses = np.array([particle["fitness"] for particle in optimizer.particles])
    fitness_range = np.ptp(fitnesses)
    if np.isclose(fitness_range, 0.0):
        normalized_fitnesses = np.zeros_like(fitnesses)
    else:
        normalized_fitnesses = (fitnesses - fitnesses.min()) / fitness_range

    scatter = ax2.scatter(
        q_positions,
        r_positions,
        c=normalized_fitnesses,
        cmap="viridis",
        alpha=0.75,
        s=60,
        edgecolors="black",
        label="Final particle positions",
    )
    ax2.scatter(
        result["Q"],
        result["r"],
        color="red",
        s=200,
        marker="*",
        edgecolors="black",
        linewidth=1,
        label="Best position found",
        zorder=5,
    )
    ax2.set_xlabel("Order quantity Q (units)")
    ax2.set_ylabel("Reorder point r (units)")
    ax2.set_title("Final Swarm Positions")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    colorbar = plt.colorbar(scatter, ax=ax2)
    colorbar.set_label("Normalized final-particle cost (lower is better)")

    target_sl = optimizer.service_level * 100
    achieved_sl = result["service_level_achieved"] * 100
    categories = ["Target", "Achieved", "Gap"]
    values = [target_sl, achieved_sl, achieved_sl - target_sl]
    colors = [
        "lightblue",
        "lightgreen",
        "lightcoral" if achieved_sl < target_sl else "seagreen",
    ]
    bars = ax3.bar(categories, values, color=colors, edgecolor="black")
    ax3.set_ylabel("Cycle service level (%)")
    ax3.set_title("Target, Achieved Level, and Gap")
    ax3.axhline(y=0, color="black", linewidth=0.5)
    for bar, value in zip(bars, values):
        height = bar.get_height()
        offset = 0.02 if height >= 0 else -0.02
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            f"{value:+.3f}%",
            ha="center",
            va="bottom" if height >= 0 else "top",
        )

    components = optimizer.calculate_cost_components(result["Q"], result["r"])
    component_names = [
        "Holding",
        "Ordering",
        "Stockout proxy",
        "Service penalty",
    ]
    component_values = [
        components["holding_cost"],
        components["ordering_cost"],
        components["stockout_cost"],
        components["service_penalty"],
    ]
    ax4.barh(component_names, component_values, color=["#ff6b6b", "#4ecdc4", "#45b7d1", "#845ec2"])
    ax4.set_xlabel("Modeled annual cost ($)")
    ax4.set_title(f"Cost Components (total: ${result['total_cost']:,.2f})")
    ax4.grid(True, axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()


create_pso_visualization(result, optimizer)

In [ ]:
def compare_final_swarm(pso_result: Dict, optimizer: PSOInventoryOptimizer) -> pd.DataFrame:
    """Summarize the best-found policy and the distribution of final particle costs."""
    final_costs = np.array([particle["fitness"] for particle in optimizer.particles])
    comparison_df = pd.DataFrame(
        [
            {
                "Candidate": "Best position found during search",
                "Q": pso_result["Q"],
                "r": pso_result["r"],
                "Safety_Stock": pso_result["safety_stock"],
                "Modeled_Annual_Cost": pso_result["total_cost"],
                "Cycle_Service_Level": pso_result["service_level_achieved"],
            },
            {
                "Candidate": "Median final-particle cost",
                "Q": np.nan,
                "r": np.nan,
                "Safety_Stock": np.nan,
                "Modeled_Annual_Cost": np.median(final_costs),
                "Cycle_Service_Level": np.nan,
            },
        ]
    )

    print("=== BEST-FOUND POLICY AND FINAL-SWARM COST SUMMARY ===")
    print(comparison_df.round(4).to_string(index=False))
    print(
        "The final-particle median is a within-run search summary; "
        "it is not a separate optimization method."
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(final_costs, bins=min(12, len(final_costs)), color="#3498db", edgecolor="black")
    axes[0].axvline(
        pso_result["total_cost"],
        color="red",
        linestyle="--",
        label="Best cost found during search",
    )
    axes[0].set_xlabel("Final-particle modeled annual cost ($)")
    axes[0].set_ylabel("Number of particles")
    axes[0].set_title("Distribution of Final-Particle Costs")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(
        [particle["position"][0] for particle in optimizer.particles],
        [particle["position"][1] for particle in optimizer.particles],
        color="#3498db",
        alpha=0.7,
        edgecolors="black",
        label="Final particles",
    )
    axes[1].scatter(
        pso_result["Q"],
        pso_result["r"],
        color="red",
        marker="*",
        s=200,
        edgecolors="black",
        label="Best position found",
        zorder=5,
    )
    axes[1].set_xlabel("Order quantity Q (units)")
    axes[1].set_ylabel("Reorder point r (units)")
    axes[1].set_title("Final Swarm in the Search Space")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return comparison_df


swarm_comparison = compare_final_swarm(result, optimizer)

In [ ]:
def parameter_sensitivity_analysis(
    demand_params: DemandParameters,
    cost_params: CostParameters,
    service_level: float,
) -> pd.DataFrame:
    """Examine how selected swarm sizes affect the stochastic PSO search."""
    swarm_sizes = [20, 40, 60]
    results = []

    for swarm_size in swarm_sizes:
        print(f"\nTesting swarm size: {swarm_size}")
        test_optimizer = PSOInventoryOptimizer(
            demand_params=demand_params,
            cost_params=cost_params,
            service_level=service_level,
            swarm_size=swarm_size,
            max_iterations=100,
        )
        test_result = test_optimizer.optimize()

        results.append(
            {
                "Swarm_Size": swarm_size,
                "Modeled_Annual_Cost": test_result["total_cost"],
                "Iterations": test_result["iterations"],
                "Early_Stop": test_result["converged"],
                "Cycle_Service_Level": test_result["service_level_achieved"],
                "Q": test_result["Q"],
                "r": test_result["r"],
            }
        )

        print(f"  Best modeled cost: ${test_result['total_cost']:,.2f}")
        print(f"  Iterations performed: {test_result['iterations']}")
        print(f"  Early stopping criterion met: {test_result['converged']}")

    sensitivity_df = pd.DataFrame(results)
    print("\n=== SWARM-SIZE SENSITIVITY ANALYSIS ===")
    print(sensitivity_df.round(4).to_string(index=False))
    print(
        "Each run is stochastic, so differences reflect both swarm size "
        "and random search variation."
    )

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("PSO Sensitivity to Swarm Size", fontsize=14, fontweight="bold")

    ax1.plot(
        sensitivity_df["Swarm_Size"],
        sensitivity_df["Modeled_Annual_Cost"],
        "bo-",
        linewidth=2,
        markersize=8,
    )
    ax1.set_xlabel("Swarm size (particles)")
    ax1.set_ylabel("Best modeled annual cost ($)")
    ax1.set_title("Best Cost by Swarm Size")
    ax1.grid(True, alpha=0.3)

    ax2.plot(
        sensitivity_df["Swarm_Size"],
        sensitivity_df["Iterations"],
        "ro-",
        linewidth=2,
        markersize=8,
    )
    ax2.set_xlabel("Swarm size (particles)")
    ax2.set_ylabel("Iterations performed")
    ax2.set_title("Iterations by Swarm Size")
    ax2.grid(True, alpha=0.3)

    ax3.plot(
        sensitivity_df["Swarm_Size"],
        sensitivity_df["Cycle_Service_Level"] * 100,
        "go-",
        linewidth=2,
        markersize=8,
        label="Achieved",
    )
    ax3.axhline(
        y=service_level * 100,
        color="red",
        linestyle="--",
        label=f"Target ({service_level * 100:.1f}%)",
    )
    ax3.set_xlabel("Swarm size (particles)")
    ax3.set_ylabel("Cycle service level (%)")
    ax3.set_title("Service Level by Swarm Size")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    for _, row in sensitivity_df.iterrows():
        ax4.scatter(
            row["Q"],
            row["r"],
            s=100,
            alpha=0.7,
            edgecolors="black",
            label=f"Swarm size {int(row['Swarm_Size'])}",
        )
    ax4.set_xlabel("Order quantity Q (units)")
    ax4.set_ylabel("Reorder point r (units)")
    ax4.set_title("Best-Found Positions")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return sensitivity_df


sensitivity_results = parameter_sensitivity_analysis(
    demand_params=demand_params,
    cost_params=cost_params,
    service_level=service_level,
)

In [ ]:
print("=== PARTICLE SWARM OPTIMIZATION SUMMARY ===")
print("\nALGORITHM SETTINGS:")
print(f"  Swarm size: {optimizer.swarm_size} particles")
print(f"  Maximum iterations: {optimizer.max_iterations}")
print(f"  Iterations performed: {result['iterations']}")
print(f"  Early stopping criterion met: {result['converged']}")
print(f"  Inertia weight range: {optimizer.w_max:.1f} to {optimizer.w_min:.1f}")
print(f"  Cognitive coefficient: {optimizer.c1:.1f}")
print(f"  Social coefficient: {optimizer.c2:.1f}")

print("\nBEST POLICY FOUND IN THIS RUN:")
print(f"  Order quantity Q: {result['Q']:.2f} units")
print(f"  Reorder point r: {result['r']:.2f} units")
print(f"  Safety stock: {result['safety_stock']:.2f} units")
print(f"  Modeled annual cost: ${result['total_cost']:,.2f}")
print(
    f"  Achieved cycle service level: "
    f"{result['service_level_achieved'] * 100:.3f}%"
)
print(f"  Target cycle service level: {service_level * 100:.3f}%")

print("\nINTERPRETATION:")
print(
    "  The reported policy is the lowest-cost candidate encountered by "
    "this stochastic swarm search."
)
print(
    "  Meeting the stopping criterion indicates limited recent improvement; "
    "it does not prove global optimality."
)
print(
    "  The service-level target is represented by a soft penalty, and the "
    "stockout-cost term is an approximation."
)
print(
    "  Run-to-run results can vary because the particle positions and "
    "update coefficients are randomized."
)
print("\nParticle swarm optimization run complete.")